# Environment Check

## What you will learn
How to verify a classroom runtime, distinguish core dependencies from optional research packages, and write a first artifact.

## Codes used
Core Python stack; optional imports for JAX, VMEC-JAX, Boozer, NEO, SFINCS-JAX, SPECTRAX-GK, SIMSOPT, NEOPAX, and ESSOS.

## Run mode
This notebook uses RUN_MODE = "cached" by default. Allowed values are "tiny", "cached", and "research".

## Expected outputs
Package-status table, backend notes, folder checks, and `assets/figures/environment_check.png`.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src" / "sos2026").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    print("Colab detected. Keep RUN_MODE='cached' first; install requirements-colab.txt from the cloned repo if needed.")
else:
    print("Local runtime detected.")

In [ ]:
RUN_MODE = "cached"  # allowed: "tiny", "cached", "research"
print(f"RUN_MODE = {RUN_MODE}")

In [ ]:
import importlib
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sos2026.paths import PROJECT_ROOT, DATA_DIR, CACHE_DIR, FIGURE_DIR, MOVIE_DIR, ensure_directories
ensure_directories()
print("Figures:", FIGURE_DIR)
print("Cached data:", CACHE_DIR)

## 1. Why this notebook exists

A school repository must survive many laptops. The scientific stack is valuable, but the lecture should not fail if a compiled package is absent. This first notebook separates the robust teaching layer from optional research packages.

In [ ]:
import platform
print("Python", platform.python_version())
print("Platform", platform.platform())

## 2. Core packages are required

These packages define cached mode. If any of them fail, the classroom environment is not ready.

In [ ]:
core_modules = ["numpy", "scipy", "matplotlib", "pandas", "xarray", "netCDF4", "h5py", "nbformat", "nbconvert", "pytest"]
rows = []
for name in core_modules:
    try:
        module = importlib.import_module(name)
        rows.append({"module": name, "required": True, "status": "OK", "version": getattr(module, "__version__", "unknown")})
    except Exception as exc:
        rows.append({"module": name, "required": True, "status": "FAIL", "version": f"{type(exc).__name__}: {exc}"})
core_df = pd.DataFrame(rows)
core_df

## 3. Optional packages are research accelerators

A missing optional package is not a lecture failure. It only decides whether a notebook can use `research` mode.

In [ ]:
optional_modules = {
    "JAX": "jax",
    "vmec_jax": "vmec_jax",
    "booz_xform_jax": "booz_xform_jax",
    "NEO_JAX": "neo_jax",
    "sfincs_jax": "sfincs_jax",
    "SPECTRAX-GK": "spectrax",
    "SIMSOPT": "simsopt",
    "NEOPAX": "NEOPAX",
    "ESSOS": "essos",
}
rows = []
for label, module_name in optional_modules.items():
    try:
        module = importlib.import_module(module_name)
        rows.append({"package": label, "import": module_name, "status": "OK", "version": getattr(module, "__version__", "unknown")})
    except Exception as exc:
        rows.append({"package": label, "import": module_name, "status": "missing/fail", "version": type(exc).__name__})
optional_df = pd.DataFrame(rows)
optional_df

## 4. JAX backend check

A laptop may have CPU-only JAX, GPU JAX, or no JAX. The backend matters because the first live run can trigger compilation.

In [ ]:
try:
    import jax
    print("JAX backend:", jax.default_backend())
    print("JAX devices:", jax.devices())
except Exception as exc:
    print("JAX unavailable or failed to initialize:", exc)

## 5. Required folder check

The repo uses fixed folders so notebooks can write the same outputs on local machines and CI.

In [ ]:
folders = [DATA_DIR, CACHE_DIR, FIGURE_DIR, MOVIE_DIR, PROJECT_ROOT / "notebooks", PROJECT_ROOT / "slides"]
folder_df = pd.DataFrame({"path": [str(p.relative_to(PROJECT_ROOT)) for p in folders], "exists": [p.exists() for p in folders]})
folder_df

## 6. Minimal plot write test

This confirms that matplotlib can write a file where the slides expect it.

In [ ]:
x = np.linspace(0, 1, 160)
fig, ax = plt.subplots(figsize=(6.2, 3.6))
ax.plot(x, np.sin(2*np.pi*x), lw=2.5, color="#0f766e")
ax.set_title("Environment check plot")
ax.set_xlabel("x")
ax.set_ylabel("sin(2 pi x)")
ax.grid(alpha=0.25)
path = FIGURE_DIR / "environment_check.png"
fig.savefig(path, dpi=160, bbox_inches="tight")
plt.show()
print("Caption: this small figure proves that cached-mode plotting works in this environment.")
print(path)

## 7. Decision rule for the rest of the course

If core packages pass, use cached mode. If optional packages pass and the instructor has tested runtime, selected demos may switch to `tiny` or `research` mode.

In [ ]:
summary = pd.DataFrame({
    "layer": ["core cached mode", "optional research mode"],
    "pass_condition": ["all core imports OK", "selected package imports and runtime tests OK"],
    "classroom_action": ["proceed", "use only if pre-tested"]
})
summary

## Try this
Change one optional import name to a package you know is absent and rerun the optional table.

## Expected qualitative answer
The optional table should report a failure without breaking cached mode.

## Research extension
Add a package-specific runtime smoke test, but keep it non-fatal for students.